# Geospatial visualization of routes

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, MultiPoint, LineString

import folium
# from folium.plugins import Legend   # having trouble on the import, using branca instead
from folium.features import GeoJsonPopup

import branca


In [ ]:
%%time

# Set some pandas options for tables
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

## Load data and merge CSV attribute data with geospatial data

In [ ]:
%%time

# Load TBI CSV data and the four geopackage data files of routes by mode
# May take ~10 minutes but once data is loaded the dataframes are fast for querying and plotting

df_tbi = pd.read_csv('data/tbi_merged.csv', low_memory=False)      
gdf_walk_trips = gpd.read_file('data/walk_trips.gpkg')              
gdf_bike_trips = gpd.read_file('data/bike_trips.gpkg')
gdf_transit_trips = gpd.read_file('data/transit_trips.gpkg')
gdf_car_trips = gpd.read_file('data/car_trips.gpkg')


In [ ]:
%%time 

# Merge the tbi attribute data with the geopackage data

gdf_walk_merged = gdf_walk_trips.merge(df_tbi, on='trip_id')
gdf_bike_merged = gdf_bike_trips.merge(df_tbi, on='trip_id')
gdf_transit_merged = gdf_transit_trips.merge(df_tbi, on='trip_id')
gdf_car_merged = gdf_car_trips.merge(df_tbi, on='trip_id')

In [ ]:
# Location data is stored as a JSON with lon and lat property values, so
# we'll convert those features to a geodataframe using Shapely

# Load the CSV
df_location = pd.read_csv('data/location.csv', low_memory=False)  

# Create a Shapely Point geometry column from the latitude and longitude columns
geometry = [Point(xy) for xy in zip(df_location['lon'], df_location['lat'])]

# Create a GeoDataFrame from the DataFrame and the geometry column
gdf_location = gpd.GeoDataFrame(df_location, geometry=geometry)

# Set the coordinate reference system (CRS) to WGS84 (EPSG:4326)
gdf_location = gdf_location.set_crs(epsg=4326)

In [ ]:
gdf_location.head()

In [ ]:
gdf_location.info()

In [ ]:
# Merge TBI CSV data with the location data -- need to debug ValueError, 
# geopackage trip_id is string representation of list while gdf_location is int64

gdf_location_merged = gdf_location.merge(df_tbi, on='trip_id')

## Mapping visualization

In [ ]:
# Categorical color scheme for various travel modes
colorMap = {
    'bike': '#00008B',      # darkblue
    'walk': '#006400',      # darkgreen
    'transit': '#4B0082',   # indigo
    'car': '#8B4513',       # saddlebrown
    'location':'#E46707'    # orange
}

# Object for storing attributes used to filter datasets
attributeMap = {
    'Mode of route': 'mode',
    'Origin purpose': 'o_purpose',
    'Destination purpose': 'd_purpose',
    'Number of travelers': 'num_travelers',
    'Distance': 'distance',
    'Speed (MPH)': 'speed_mph',
    'VMT': 'vmt',
    'Age': 'age'
}

# conditions for evaluating query expressions
conditions = ['==', '!=', '<', '<=', '>', '>=']

In [ ]:
def select_random_trip_id(gdf):
    # function accepts a geodataframe, selects a random row
    # checks to see if the tripId contains more than one value (inferred from length of string)
    # returns string representation of trip id

    row = gdf.sample(n=1)
    trip_id = row['trip_id'].iloc[0][1:-1]   # strip out brackets
    
    # if the trip_id contains more than one trip
    if(len(tripId) > 13):
        trip_id = tripId.split(',')
        return  trip_id[0]
    else:
        return trip_id


## FOR REFERENCE: length of trip_id values within each dataset, includes brackets (i.e., number of trips)

# long = gdf_walk_merged[gdf_walk_merged['trip_id'].str.len() > 15]
# longIndex = long.sample().index

# print(gdf_walk_merged.iloc[longIndex])


# bikes
# length count
# 15    247004
# 30      6044
# 45       868
# 60       183
# 75        54
# 90        14

# walk
# 15     444679
# 30       9848
# 45       3246
# 60        472
# 75        306
# 90         53
# 105        16
# 135         9
# 120         7
# 150         2

# car
# 15     444678
# 30       9848
# 45       3246
# 60        472
# 75        306
# 90         53
# 105        16
# 135         9
# 120         7
# 150         2

# transit
# 15     778124
# 30      17772
# 45      10701
# 60       1740
# 75       1237
# 90        247
# 105        55
# 135        31
# 120        24
# 150         6

In [ ]:

def map_trip(trip_id):
    gdf_walk_trip = gdf_walk_merged[gdf_walk_merged['trip_id'].str.contains(trip_id)]
    gdf_bike_trip = gdf_bike_merged[gdf_bike_merged['trip_id'].str.contains(trip_id)]
    gdf_car_trip = gdf_car_merged[gdf_car_merged['trip_id'].str.contains(trip_id)]
    gdf_transit_trip = gdf_transit_merged[gdf_transit_merged['trip_id'].str.contains(trip_id)]
    # gdf_location_trip = gdf_location_merged[gdf_location_merged['trip_id'].str.contains(trip_id)] # need to fix merge

    # tooltip current broken when adding as arugment below in folium.GeoJson
    tooltip=folium.features.GeoJsonTooltip(fields=['duration_seconds', 'duration_meters', 'weight', 'mode', 'o_purpose', 'd_purpose', 'num_travelers', 'distance', 'speed_mph', 'vmt', 'age'],
                                                      aliases=['Duration (sec)', 'Duration (meters)', 'Weight', 'Mode', 'O purpose', 'D purpose', '# of travelers', 'Distance', 'Speed (mph)', 'VMT', 'Age'])
    
    # instantiate a global Leaflet map object using Folium
    map = folium.Map(location=[44.9778,-93.2650], tiles="CartoDB positron", zoom_start=10)
    folium.GeoJson(gdf_walk_trip, name='Walk route', style_function=lambda feature: {'color':colorMap['walk']}).add_to(map)
    folium.GeoJson(gdf_bike_trip, name='Bike route', style_function=lambda feature: {'color':colorMap['bike']}).add_to(map)
    folium.GeoJson(gdf_car_trip, name='Car route', style_function=lambda feature: {'color':colorMap['car']}).add_to(map)
    folium.GeoJson(gdf_transit_trip, name='Transit route', style_function=lambda feature: {'color':colorMap['transit']}).add_to(map)

    # Need to add location gdf when merge is solved above

    # # create a legend for each unique color
    # # currently broken on import, using alternative legend solution below
    # legend = Legend(position='bottomright', title='Route Mode', colors=[colorMap['walk'], colorMap['bike'],colorMap['car'],colorMap['transit']], labels=['Walking', 'Biking', 'Driving', 'Transit'])
    # legend.add_to(map)

    # create html string for legend
    legend_html = '''
    <div style="position: fixed; 
        bottom: 50px; right: 50px; width: 150px; height: 100px; 
        border:2px solid grey; z-index: 9999; font-size:14px;">
        &nbsp;<b>Legend</b><br>
        &nbsp;<i class="fa fa-circle" style="color:#006400"></i>&nbsp;Walking<br>
        &nbsp;<i class="fa fa-circle" style="color:#00008B"></i>&nbsp;Biking<br>
        &nbsp;<i class="fa fa-circle" style="color:#8B4513"></i>&nbsp;Driving<br>
        &nbsp;<i class="fa fa-circle" style="color:#4B0082"></i>&nbsp;Transit
    </div>
    '''
    
    # use branca to add legend to the map
    legend = branca.element.Element(legend_html)
    map.get_root().html.add_child(legend)


    route_bounds = gdf_walk_trip.bounds
    map.fit_bounds([[route_bounds['miny'].values[0], route_bounds['minx'].values[0]], [route_bounds['maxy'].values[0], route_bounds['maxx'].values[0]]])

    # add a legend to the map
    folium.LayerControl().add_to(map)
    return map


  
    
    # function accepts a unique trip ID as a string and plots trips on a map
    # for each of 5 routes different color
    # fits bounds of map to mapped features
        # provide tooltip/popup:
            # include all attributes from geopackage
            # include from tbi_merged:
                # mode, o_purpose, d_purpose, num_travelers, distance, speed_mph, vmt, age



In [ ]:
# use one of the mode routes to select a random trip ID and create a map
map = map_trip(select_random_trip_id(gdf_walk_merged))
map

In [ ]:
def map_filtered_routes(filterAttribute, filterCondition, filterValue):
    # function accepts three parameters to create a expression to evaluate
    # and returns a geodataframe of matching features
    # probably want to return all geodataframes as a tuple
    expression = f'{filterAttribute} {filterCondition} {filterValue}'
    gdf_query =  gdf_transit_merged.query(expression)
    return gdf_query

In [ ]:
filterAttribute = 'speed_mph'
filterCondition = '<'
filterValue = '1'

gdf_filtered = map_filtered_routes(filterAttribute, filterCondition, filterValue)
gdf_filtered.head()

## Interactive widgets for querying/filtering data

needs further dev ...

In [ ]:
import ipywidgets as widgets

In [ ]:
# Assume `gdf` is your geodataframe with the attribute you want to select
attribute = widgets.Dropdown(
    options=list(gdf_bike_merged.columns),
    description='Attribute:'
)

# Define the condition choices
condition = widgets.Dropdown(
    options=['==', '!=', '<', '<=', '>', '>='],
    description='Condition:'
)

# Define a callback function that updates the value choices based on the selected attribute
def update_value_choices(*args):
    selected_attr = attribute.value
    value_choices.options = list(gdf_bike_merged[selected_attr].unique())

# Assume `gdf` is your geodataframe with the attribute you want to select
value_choices = widgets.Dropdown(
    options=list(gdf_bike_merged[list(gdf_bike_merged.columns)[0]].unique()),
    description='Value:'
)

# Call the `update_value_choices` function when the attribute is changed
attribute.observe(update_value_choices, 'value')

# Display the UI elements
widgets.VBox([attribute, condition, value_choices])

## scraps

In [ ]:
# Assume `gdf` is your geodataframe with the attribute you want to select
attribute = input(f"Select an attribute from the following: {list(gdf_bike_merged.columns)}\n")

# Define the condition choices
condition_choices = ['==', '!=', '<', '<=', '>', '>=']
condition = input(f"Select a condition from the following: {condition_choices}\n")

# Assume `gdf` is your geodataframe with the attribute you want to select
selected_attr_values = list(gdf_bike_merged[attribute].unique())
value_choices_str = ", ".join(map(str, selected_attr_values))
value = input(f"Select a value from the following: {value_choices_str}\n")

# Display the selected choices
print(f"You selected: {attribute} {condition} {value}")

In [ ]:
map = folium.Map(location=[44.9778,-93.2650], tiles="CartoDB positron", zoom_start=10)

# Extract the Point objects given a specific trip ID
gdf_location_selected = gdf_location[gdf_location['trip_id'] == 1811424401001]

# Extract the Point objects from the geometry column of the GeoDataFrame
points = gdf_location_selected.geometry.values.tolist()

# Create a LineString object from the points
line = LineString(points)

# Create a MultiPoint object from the points
multipoint = MultiPoint(points)

# Create a new GeoDataFrame with the LineString as its geometry
gdf_line = gpd.GeoDataFrame(geometry=[line])

# Create a GeoDataFrame from the MultiPoint object with a column indicating if a point is an endpoint
gdf_multipoint = gpd.GeoDataFrame(geometry=[multipoint])
gdf_multipoint['is_endpoint'] = gdf_multipoint.intersects(line)

# Set the coordinate reference system (CRS) of the new GeoDataFrame to match the original
gdf_line.crs = gdf_multipoint.crs = gdf_location.crs

# Add the points as a GeoJSON layer with circles larger than the width of the line
style_function = lambda x: {'color': 'black', 'weight': 2, 'fillOpacity': 0.8, 'radius': 8}
folium.GeoJson(gdf_location_selected.to_json(), style_function=style_function).add_to(map)

# Add the LineString as a GeoJSON layer with a dotted orange line and width of 2
style_function = lambda x: {'color': 'orange', 'dashArray': '3,3', 'weight': 2}
folium.GeoJson(gdf_line.__geo_interface__, style_function=style_function).add_to(map)

# Add the endpoint markers as a separate GeoJSON layer with larger red circles
style_function = lambda x: {'color': 'red', 'fillOpacity': 0.8, 'radius': 12 if x['properties']['is_endpoint'] else 6}
folium.GeoJson(gdf_multipoint.to_json(), style_function=style_function).add_to(map)

# Add a polyline connecting the points with an orange dotted line
folium.PolyLine(gdf_line, color='orange', weight=2, dash_array='10,10').add_to(map)

# row_bounds = gdf_location_selected.bounds
# map.fit_bounds([[row_bounds['miny'].values[0], row_bounds['minx'].values[0]], [row_bounds['maxy'].values[0], row_bounds['maxx'].values[0]]])

# gdf_location_selected.explore(map, show_bbox=False)
map